# 🛡️ Xibalba Integrity Protocol SDK
### **Developer & Swarm Operations Notebook**

This notebook serves as the official interactive demonstration and playground for the **Integrity Protocol SDK** (v0.2.0). The SDK acts as the local **High-Fidelity Measurement Apparatus** that secures and binds autonomous AI agent behavior to institutional accountability and compliance frameworks.

---

## 🏗️ Core Architectural Pillars
1. **Decentralized Agent Identity (DID)**: Securing hardware-bound cryptographic identities for each running agent instance.
2. **Behavioral Commitment Chain (BCC)**: Cryptographic intent anchoring, replay protection, and drift detection.
3. **Zero-Trust Telemetry Ingestion**: Async batching and streaming of model cognitive metrology and system host telemetry.
4. **Offline Cache Moat**: HMAC-SHA256 authenticated local SQLite fallback storage ensuring zero data loss during network disruptions.
5. **Compliance-as-Code**: HIPAA-eligible sandbox boundaries and Zero Data Retention (ZDR) verification.

---
## ⚙️ Setup & Imports
Ensure that the `integrity_sdk` package path is appended to the system search path, and import the core classes.

In [ ]:
import os
import sys
import time
import json
import sqlite3

# Append local Projects directory to path to ensure importing from latest source
sdk_root = os.path.abspath("/home/xibalba/Projects/integrity-sdk")
if sdk_root not in sys.path:
    sys.path.insert(0, sdk_root)

from integrity_sdk import IntegrityClient
from integrity_sdk.client import BCCCommitment

print("✅ Setup completed. Integrity SDK imported successfully.")

---
## 🆔 Task 1: Decentralized Agent Identity (DID) & Hardware Binding
When initializing an `IntegrityClient`, a deterministic DID keypair is loaded or generated based on the agent's ID. This keypair is cryptographically bound to the local host's hardware signature.

In [ ]:
# Initialize a trading agent client
client_alpha = IntegrityClient(
    agent_id="agent_trader_alpha",
    oracle_url="http://localhost:3001/ingest"
)

print(f"🔹 Agent ID:          {client_alpha.agent_id}")
print(f"🔑 W3C DID:           {client_alpha.did}")
print(f"💳 EVM Wallet:        {client_alpha.wallet_address}")
print(f"💻 Hardware ID Hash:  {client_alpha.hardware_fingerprint}")

---
## 📈 Task 2: High-Fidelity Telemetry Ingestion & Metrology Heuristics
The SDK captures and computes cognitive safety and performance heuristics (Type-Token Ratio for vocabulary diversity, mean token confidence, perplexity, and structural format compliance) before batching and streaming to the Oracle.

In [ ]:
# Log a mock model inference output
mock_inference_metadata = {
    "model_name": "gemini-3.5-flash",
    "provider": "Google",
    "prompt_text": "Calculate portfolio variance for BTC and ETH.",
    "text_output": "Portfolio variance is minimized at a 60/40 allocation weight split.",
    # Simulated token logprobabilities
    "token_logprobs": [-0.05, -0.1, -0.02, -0.15, -0.01],
    "gpu_hours_used": 0.0015
}

print("🚀 Logging model telemetry...")
client_alpha.log_telemetry(metadata=mock_inference_metadata)

# Check what metrics were dynamically computed by the metrology engine
print("\n📊 Dynamic Metrology Metrics Computed:")
print(f"  - Vocab Diversity (TTR):      {mock_inference_metadata.get('vocabulary_diversity')}")
print(f"  - Mean Token Confidence:      {mock_inference_metadata.get('mean_token_confidence')}%" if 'mean_token_confidence' in mock_inference_metadata else "  - Mean Token Confidence: (N/A)")
print(f"  - Perplexity:                 {mock_inference_metadata.get('perplexity')}")

---
## 🔗 Task 3: Behavioral Commitment Chain (BCC) & Intent Drift Prevention
The BCC prevents arbitrary or drifted model executions. The agent first registers a signed *pre-commitment* of its intended state and policy parameters, then executes the function under validation.

In [ ]:
# Define a mock action function
def execute_portfolio_rebalance(asset, target_percentage):
    print(f"💼 Execution Action: Rebalancing {asset} to {target_percentage}%")
    return "SUCCESS"

# 1. Define intended state
intended_state = {
    "asset": "BTC",
    "target_percentage": 65.0
}

# 2. Generate cryptographically signed commitment
print("🔒 Generating BCC commitment...")
commitment = client_alpha.commit_action_intent(
    action_type="portfolio_rebalance",
    intended_state=intended_state,
    opa_policy_id="policy_max_allocation_limit"
)

print(f"  - Commitment UUID: {commitment.id}")
print(f"  - Intent Hash:      {commitment.intended_state_hash}")
print(f"  - Signature:        {commitment.provenance_signature[:60]}...")

# 3. Validate and execute (Succeeds because actual context matches commitment)
print("\n✅ Validating and executing matching context...")
client_alpha.validate_and_execute(
    commitment=commitment,
    actual_execution_context=intended_state,
    action_function=lambda: execute_portfolio_rebalance("BTC", 65.0)
)

### ⚠️ Intent Drift Simulation
If the actual execution environment attempts to deviate from the pre-committed state (e.g. changing the asset allocation weight due to drift or context hijack), the SDK blocks execution.

In [ ]:
# Drifted state (changing target allocation to 95.0% without generating a new commitment)
drifted_state = {
    "asset": "BTC",
    "target_percentage": 95.0
}

try:
    print("🚨 Validating drifted context (should raise exception)...")
    client_alpha.validate_and_execute(
        commitment=commitment,
        actual_execution_context=drifted_state,
        action_function=lambda: execute_portfolio_rebalance("BTC", 95.0)
    )
except RuntimeError as err:
    print(f"\n❌ BCC Blocked Execution: {err}")

---
## 🏥 Task 4: HIPAA Compliance-as-Code & Sandboxing
For agents handling sensitive data (e.g., healthcare scribes), the client is configured as HIPAA-eligible, enabling special boundary audit logging and ZDR checks.

In [ ]:
# Initialize a clinical agent
client_beta = IntegrityClient(
    agent_id="agent_clinician_beta",
    hipaa_eligible=True,
    zdr_enabled=True
)

print(f"🏥 Clinician Agent HIPAA Enabled: {client_beta.hipaa_eligible}")
print(f"🛡️ Zero Data Retention (ZDR) Enforced: {client_beta.zdr_enabled}")

# Log compliance audit event
client_beta.log_compliance_event(
    event_type="hipaa_shield_activated",
    status="success",
    details="Clinical sandbox boundary validated for US-EAST-1"
)

---
## 💾 Task 5: Offline Cache Moat & HMAC Tamper-Proofing
To guarantee **zero data loss** if the Oracle is offline or unreachable, the client redirects all queued telemetry into an HMAC-SHA256 authenticated local SQLite database.

In [ ]:
# Resolve the local SQLite cache path for client_alpha
db_dir = os.path.expanduser("~/.integrity")
db_path = os.path.join(db_dir, f"offline_moat_{client_alpha.agent_id}.db")

print(f"📁 Local Cache DB Path: {db_path}")

# Log some offline actions
client_alpha.log_telemetry(metadata={"offline_event": "rebalance_offline_exec"})

# Flush cache manually or let background thread flush
time.sleep(2.0) # Wait for batching to register

# Query the local cache database directly using sqlite3 to view rows
if os.path.exists(db_path):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute("SELECT id, timestamp, payload, hmac_signature FROM telemetry_cache LIMIT 5")
    rows = cursor.fetchall()
    
    print(f"\n🔍 Found {len(rows)} cached rows in local SQLite offline moat:")
    for r in rows:
        row_id, ts, payload, sig = r
        print(f"  - ID:        {row_id}")
        print(f"    Payload:   {payload[:90]}...")
        print(f"    HMAC Sig:  {sig[:50]}...")
    conn.close()
else:
    print("⚠️ Local offline database cache file does not exist yet (no unsynced offline records in buffer).")

---
## 🐝 Task 6: Multi-Agent Swarms & Isolated Workspaces
Swarms of concurrent agents can be spawned from a parent configuration to maintain workspace isolation.

In [ ]:
# Spawn child agent from parent clinician client
child_scribe = client_beta.spawn_subagent(subagent_id="scribe_subagent_01")

print(f"👨‍👦 Parent Agent ID:     {client_beta.agent_id}")
print(f"👦 Subagent Namespace:   {child_scribe.subagent_id}")
print(f"🔑 Subagent DID Inherited: {child_scribe.did}")